# Phase 2b: Retuned Topology + Burial-Control Analysis

Two jobs, one run:

1. **Fix the dead features.** Phase 2 sampled Betti curves at 4/6/8 Å, but in a
   12 Å neighbourhood every $H_1$/$H_2$ bar is dead well before 8 Å, five
   columns came out constant. Radii move to **2/3/5 Å**, and any column that is
   still constant is pruned automatically rather than by hand.

2. **Answer the question the whole project rests on.** Phase 2's strongest
   single feature was `nbhd_gyration` (AUC 0.722), which is *not topological*.
   It is a burial/density proxy, and burial has been a strong binding-site
   baseline for decades. Before claiming persistent homology contributes
   anything, we have to show it adds signal **beyond burial**.

---

### How the claim is tested

**Residualisation.** Regress each topological feature on the burial features
(`nbhd_n_atoms`, `nbhd_gyration`, plus their squares and interaction, so burial
gets a *non-linear* chance to explain the feature), then score the residual
against the labels. A feature whose residual AUC collapses to 0.5 was measuring
burial all along.

**Nested models with grouped CV.** Five feature sets, `GroupKFold` grouped by
PDB entry so no protein appears in both train and test. The decisive number is
the *marginal* gain of adding topology to everything else:

| set | features |
|---|---|
| `seq` | Phase 1 one-hot + dihedrals (27) |
| `burial` | `nbhd_n_atoms`, `nbhd_gyration` (2) |
| `topo` | topological only, burial excluded |
| `seq+burial` | the honest baseline to beat |
| `seq+burial+topo` | the full model |

Both a linear probe (logistic regression) and a non-linear one
(`HistGradientBoostingClassifier`) are run, because a linear probe alone would
understate topology if its contribution is non-linear.

> This is a **probe**, not the Phase 3 model. It measures whether the signal is
> present and separable, not the ceiling an EGNN could reach.

## Cell 1: Setup

In [ ]:
!pip install -q torch-geometric==2.6.1
!pip install -q gudhi==3.10.1
!pip install -q biopython==1.84

In [ ]:
import os, time, warnings
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import gudhi
import requests
from concurrent.futures import ThreadPoolExecutor
from scipy.spatial import cKDTree
from Bio.PDB import MMCIFParser, PDBParser

warnings.filterwarnings("ignore", module="Bio")
print(f"gudhi {gudhi.__version__} | torch {torch.__version__} | CPUs {os.cpu_count()}")


def locate_phase1() -> Path:
    cands = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            cands.extend(root.rglob("phase1_graphs.pt"))
    if not cands:
        raise FileNotFoundError("phase1_graphs.pt not found -- attach the Phase 1 notebook output")
    return max(cands, key=lambda p: p.stat().st_size)


blob = torch.load(locate_phase1(), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
print(f"chains {len(dataset)} | residues {sum(d.num_nodes for d in dataset):,} "
      f"| positives {sum(int(d.y.sum()) for d in dataset):,}")

## Cell 2: Heavy-atom clouds (same alignment guarantees as Phase 2)

In [ ]:
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)
AA3 = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
       "MET","PHE","PRO","SER","THR","TRP","TYR","VAL","MSE"}


def fetch(pid):
    for ext in ("pdb", "cif"):
        p = RAW / f"{pid}.{ext}"
        if p.exists() and p.stat().st_size > 1000:
            return p
    for ext in ("pdb", "cif"):
        try:
            r = requests.get(f"https://files.rcsb.org/download/{pid}.{ext}", timeout=30)
            if r.status_code == 200 and len(r.content) > 1000:
                p = RAW / f"{pid}.{ext}"; p.write_bytes(r.content); return p
        except requests.RequestException:
            continue
    return None


needed = sorted({d.pdb_id for d in dataset})
with ThreadPoolExecutor(max_workers=8) as ex:
    paths = dict(zip(needed, ex.map(fetch, needed)))
print(f"structures: {sum(v is not None for v in paths.values())}/{len(needed)}")


def chain_atoms(pid, path):
    parser = MMCIFParser(QUIET=True) if path.suffix == ".cif" else PDBParser(QUIET=True, PERMISSIVE=True)
    model = next(parser.get_structure(pid, str(path)).get_models())
    out = {}
    for ch in model:
        res_list = [r for r in ch if r.get_resname().strip().upper() in AA3 and "CA" in r]
        if not res_list:
            continue
        per_res, cas = [], []
        for res in res_list:
            xyz = [a.coord for a in res.get_unpacked_list() if a.element not in ("H", "D")]
            per_res.append(np.asarray(xyz, dtype=np.float32).reshape(-1, 3))
            cas.append(np.asarray(res["CA"].coord, dtype=np.float32))   # by NAME, not index
        out[ch.id] = (per_res, np.stack(cas))
    return out


cache = {p: chain_atoms(p, paths[p]) for p in needed if paths[p] is not None}

aligned = []
for d in dataset:
    e = cache.get(d.pdb_id, {}).get(d.chain_id)
    if e is not None and len(e[0]) == d.num_nodes:
        aligned.append((d, e[0], e[1]))
print(f"aligned chains: {len(aligned)}/{len(dataset)}")
assert len(aligned) == len(dataset), "alignment regression vs Phase 2"

d0, _, ca0 = aligned[0]
p1 = d0.pos.numpy(); p1 -= p1.mean(0); p2 = ca0 - ca0.mean(0)
drift = float(np.abs(np.linalg.norm(p1[:, None] - p1[None], axis=-1)
                     - np.linalg.norm(p2[:, None] - p2[None], axis=-1)).max())
print(f"CA geometry drift vs Phase 1: {drift:.2e} A")
assert drift < 1e-2

## Cell 3: Featuriser with retuned Betti radii

Only `BETTI_RADII` changes versus Phase 2 (4/6/8 → **2/3/5 Å**). Constant-column
pruning is now automatic and data-driven, so `H0_mean_birth`, structurally zero,
since every alpha-complex vertex is born at 0, disappears without special-casing.

In [ ]:
MAX_HOM_DIM = 2
BETTI_RADII = (2.0, 3.0, 5.0)      # retuned: H1/H2 bars are all dead by 8 A
NBHD_RADIUS = 12.0
MAX_NBHD_ATOMS = 300
FEATS_PER_DIM = 6 + 2 + len(BETTI_RADII)
TOPO_DIM = FEATS_PER_DIM * (MAX_HOM_DIM + 1) + 2

TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM + 1)
              for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                        "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]] \
             + ["nbhd_n_atoms", "nbhd_gyration"]
BURIAL_NAMES = ["nbhd_n_atoms", "nbhd_gyration"]
assert len(TOPO_NAMES) == TOPO_DIM


def _diagrams(pts):
    pts = np.ascontiguousarray(pts, dtype=np.float64)
    st, squared = None, True
    if len(pts) >= 5:
        try:
            st = gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception:
            st = None
    if st is None:
        st = gudhi.RipsComplex(points=pts, max_edge_length=float(NBHD_RADIUS)) \
                  .create_simplex_tree(max_dimension=MAX_HOM_DIM + 1)
        squared = False
    st.compute_persistence(homology_coeff_field=2, persistence_dim_max=True)
    out = {}
    for d in range(MAX_HOM_DIM + 1):
        b = np.asarray(st.persistence_intervals_in_dimension(d), dtype=np.float64).reshape(-1, 2)
        # GUDHI alpha filtration values are SQUARED circumradii -> convert to angstroms
        out[d] = np.sqrt(np.maximum(b, 0.0)) if (squared and len(b)) else b
    return out


def _vectorize(dgms, pts):
    feats = []
    for d in range(MAX_HOM_DIM + 1):
        bars = dgms.get(d, np.zeros((0, 2)))
        fin = bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1, 2)
        if len(fin) == 0:
            feats.extend([0.0] * FEATS_PER_DIM); continue
        b, dth = fin[:, 0], fin[:, 1]
        pers = np.maximum(dth - b, 0.0); tot = float(pers.sum())
        if tot > 0:
            p = pers / tot; nz = p[p > 0]; ent = float(-(nz * np.log(nz)).sum())
        else:
            ent = 0.0
        feats.extend([float(len(fin)), tot, float(pers.max()), float(pers.mean()),
                      float(pers.std()), ent, float(b.mean()), float(dth.mean()),
                      *[float(((b <= r) & (dth > r)).sum()) for r in BETTI_RADII]])
    c = pts - pts.mean(0, keepdims=True)
    feats.extend([float(len(pts)), float(np.sqrt((c ** 2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(feats, dtype=np.float32)


def topo_for_chain(ca_xyz, per_res):
    atoms = np.concatenate([a for a in per_res if len(a)], axis=0)
    tree = cKDTree(atoms)
    out = np.zeros((len(ca_xyz), TOPO_DIM), np.float32)
    for i, ca in enumerate(ca_xyz):
        idx = tree.query_ball_point(ca, NBHD_RADIUS)
        if len(idx) > MAX_NBHD_ATOMS:
            _, idx = tree.query(ca, k=MAX_NBHD_ATOMS)
        pts = atoms[np.asarray(idx, dtype=int)]
        if len(pts) >= 4:
            out[i] = _vectorize(_diagrams(pts), pts)
    return out

## Cell 4: Compute

In [ ]:
from joblib import Parallel, delayed

t0 = time.time()
topo_all = Parallel(n_jobs=max(1, os.cpu_count() or 2), verbose=1)(
    delayed(topo_for_chain)(ca, pr) for _, pr, ca in aligned)
topo_cat = np.concatenate(topo_all, axis=0)
y = np.concatenate([d.y.numpy() for d, _, _ in aligned]).astype(int)
groups = np.concatenate([[d.pdb_id] * d.num_nodes for d, _, _ in aligned])
seq_feats = np.concatenate([d.x.numpy() for d, _, _ in aligned], axis=0)

print(f"\ndone in {time.time() - t0:.1f}s | topo {topo_cat.shape} | "
      f"seq {seq_feats.shape} | positives {y.sum()}/{len(y)}")

# --- automatic constant-column pruning ---------------------------------------
keep = topo_cat.std(axis=0) > 1e-9
dropped = [TOPO_NAMES[i] for i in range(TOPO_DIM) if not keep[i]]
topo = topo_cat[:, keep]
names = [TOPO_NAMES[i] for i in range(TOPO_DIM) if keep[i]]
print(f"pruned {len(dropped)} constant column(s): {dropped}")
print(f"topological features retained: {topo.shape[1]}  (Phase 2 had 30 live of 35)")

burial_idx = [names.index(n) for n in BURIAL_NAMES]
topo_only_idx = [i for i in range(len(names)) if i not in burial_idx]

## Cell 5: Residualisation: does each feature survive burial control?

Burial is given a generous, non-linear shot at explaining every topological
feature, the design matrix includes both burial terms, their squares, and their
interaction. Whatever AUC remains in the residual is signal burial *cannot*
account for.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score

B = topo[:, burial_idx]
B_aug = np.column_stack([B, B ** 2, B[:, 0] * B[:, 1]])     # non-linear burial basis

rows = []
for j in topo_only_idx:
    col = topo[:, j].astype(np.float64)
    raw = roc_auc_score(y, col)
    resid = col - LinearRegression().fit(B_aug, col).predict(B_aug)
    res_auc = roc_auc_score(y, resid) if resid.std() > 1e-12 else 0.5
    rows.append((names[j], raw, res_auc,
                 abs(res_auc - 0.5) / max(abs(raw - 0.5), 1e-9)))

rows.sort(key=lambda r: -abs(r[2] - 0.5))
print(f"{'feature':<22}{'raw AUC':>9}{'resid AUC':>11}{'signal kept':>13}")
print("-" * 55)
for n, raw, res, kept in rows[:14]:
    print(f"{n:<22}{raw:>9.3f}{res:>11.3f}{kept:>12.0%}")

surv = [r for r in rows if abs(r[2] - 0.5) > 0.05]
print(f"\n{len(surv)}/{len(rows)} topological features keep |AUC-0.5| > 0.05 after burial control")
print(f"best surviving: {surv[0][0]} at residual AUC {surv[0][2]:.3f}" if surv
      else "NOTHING survives burial control")

## Cell 6: Nested models, grouped by PDB entry

`GroupKFold` on `pdb_id` keeps every chain of a protein on one side of the split,
so a two-chain entry like 1AKE cannot leak. **`seq+burial` → `seq+burial+topo`
is the number that decides the project's premise.**

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SETS = {
    "seq":             seq_feats,
    "burial":          topo[:, burial_idx],
    "topo":            topo[:, topo_only_idx],
    "seq+burial":      np.hstack([seq_feats, topo[:, burial_idx]]),
    "seq+burial+topo": np.hstack([seq_feats, topo]),
}

def evaluate(X, model, n_splits=5):
    roc, pr = [], []
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, groups):
        if model == "logreg":
            clf = make_pipeline(StandardScaler(),
                                LogisticRegression(max_iter=3000, class_weight="balanced"))
        else:
            clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08,
                                                 max_depth=6, random_state=0)
        clf.fit(X[tr], y[tr])
        p = clf.predict_proba(X[te])[:, 1]
        roc.append(roc_auc_score(y[te], p)); pr.append(average_precision_score(y[te], p))
    return np.array(roc), np.array(pr)


results = {}
for model in ("logreg", "hgb"):
    print(f"\n=== {model} | 5-fold GroupKFold by PDB entry ===")
    print(f"{'feature set':<18}{'ROC-AUC':>16}{'PR-AUC':>16}")
    print("-" * 50)
    for name, X in SETS.items():
        roc, pr = evaluate(X, model)
        results[(model, name)] = (roc, pr)
        print(f"{name:<18}{roc.mean():>9.3f} +-{roc.std():.3f}{pr.mean():>10.3f} +-{pr.std():.3f}")

    base_r, base_p = results[(model, "seq+burial")]
    full_r, full_p = results[(model, "seq+burial+topo")]
    dr, dp = full_r - base_r, full_p - base_p
    print(f"\n  MARGINAL GAIN FROM TOPOLOGY (paired, per fold):")
    print(f"    ROC-AUC {dr.mean():+.4f} +-{dr.std():.4f}   "
          f"folds improved: {(dr > 0).sum()}/{len(dr)}")
    print(f"    PR-AUC  {dp.mean():+.4f} +-{dp.std():.4f}   "
          f"folds improved: {(dp > 0).sum()}/{len(dp)}")

## Cell 7: Plots

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

k = min(12, len(rows))
top = rows[:k][::-1]
yy = np.arange(k)
ax[0].barh(yy - 0.2, [abs(r[1] - 0.5) for r in top], height=0.4, label="raw")
ax[0].barh(yy + 0.2, [abs(r[2] - 0.5) for r in top], height=0.4, label="after burial control")
ax[0].set_yticks(yy); ax[0].set_yticklabels([r[0] for r in top], fontsize=7)
ax[0].set_xlabel("|AUC - 0.5|"); ax[0].legend(fontsize=8)
ax[0].set_title("does topological signal survive burial control?")

labels = list(SETS.keys())
w = 0.35
for i, model in enumerate(("logreg", "hgb")):
    m = [results[(model, n)][1].mean() for n in labels]
    e = [results[(model, n)][1].std() for n in labels]
    ax[1].bar(np.arange(len(labels)) + (i - 0.5) * w, m, w, yerr=e, capsize=3, label=model)
ax[1].set_xticks(range(len(labels)))
ax[1].set_xticklabels(labels, rotation=20, ha="right", fontsize=8)
ax[1].set_ylabel("PR-AUC (grouped CV)"); ax[1].legend(fontsize=8)
ax[1].set_title("nested feature sets")
plt.tight_layout(); plt.show()

## Cell 8: Save

In [ ]:
mu, sd = topo.mean(0), topo.std(0); sd[sd < 1e-6] = 1.0
out_ds = []
for (d, _, _), t_raw in zip(aligned, topo_all):
    t = t_raw[:, keep]
    d.x_topo = torch.from_numpy(t.astype(np.float32))
    d.x = torch.cat([d.x, torch.from_numpy(((t - mu) / sd).astype(np.float32))], dim=1)
    assert torch.isfinite(d.x).all()
    out_ds.append(d)

out = Path("/kaggle/working/processed"); out.mkdir(parents=True, exist_ok=True)
p = out / "phase2b_graphs.pt"
torch.save({"dataset": out_ds, "node_feat_dim": out_ds[0].x.size(1),
            "edge_feat_dim": out_ds[0].edge_attr.size(1),
            "topo_dim": topo.shape[1], "topo_names": names,
            "burial_names": BURIAL_NAMES, "dropped_names": dropped,
            "topo_mean": mu, "topo_std": sd,
            "residual_auc": {r[0]: r[2] for r in rows},
            "cv_results": {f"{m}|{n}": (results[(m, n)][0].tolist(), results[(m, n)][1].tolist())
                           for m in ("logreg", "hgb") for n in SETS},
            "pos_weight": blob["pos_weight"], "aa_alphabet": blob["aa_alphabet"]}, p)
print(f"node feature dim : 27 -> {out_ds[0].x.size(1)}")
print(f"saved -> {p} ({p.stat().st_size / 1e6:.1f} MB)")

---

## How to read this

The decisive line is **MARGINAL GAIN FROM TOPOLOGY**: `seq+burial+topo` minus
`seq+burial`, paired per fold.

- **Positive PR-AUC gain in most folds** → persistent homology carries
  information neither sequence nor burial provides. That is the finding the
  project needs, and the fold-level consistency is what makes it defensible.
- **Gain near zero** → the honest conclusion is that burial explains the earlier
  AUCs. Report it plainly; a negative result competently demonstrated is
  stronger science than an unexamined positive, and the residualisation table
  shows exactly which descriptors were redundant.

PR-AUC matters more than ROC-AUC here: at ~9 % positives, ROC-AUC is optimistic
and PR-AUC reflects the actual retrieval problem.